In [1]:
import pandas as pd
import numpy as np

In [2]:
us_finance = pd.read_csv(r"C:\Users\benso\Downloads\Data.Gov-FY25Q44.csv",sep=";",low_memory=False)
us_finance.head()

,Fiscal Year,Unique Identifier,Deal Number,Decision,Decision Date,Effective Date,Expiration Date,Brokered,Deal Cancelled,Country,...,Working Capital Delegated Authority,Approved/Declined Amount,Disbursed/Shipped Amount,Undisbursed Exposure Amount,Outstanding Exposure Amount,Small Business Authorized Amount,Woman Owned Authorized Amount,Minority Owned Authorized Amount,Loan Interest Rate,Multiyear Working Capital Extension
0,2007,08003048XA0002,AP003048AA,Approved,11/29/2006,NaN,12/15/2011,No,No,Private Export Funding Corp.,...,NaN,24500000.0,24500000.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,2007,08003048XA0003,AP003048AA,Approved,11/29/2006,11/29/2006,12/15/2016,No,No,Private Export Funding Corp.,...,NaN,50000000.0,50000000.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,2007,08003048XA0004,AP003048AA,Approved,8/16/2007,8/16/2007,9/15/2017,No,No,Private Export Funding Corp.,...,NaN,136250000.0,136250000.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,2007,08064737XM0001,AP064737XM,Approved,9/21/2007,9/21/2007,7/31/2008,No,No,United States,...,Yes,2700000.0,2700000.0,0.0,0.0,2700000.0,0.0,2700000.0,NaN,NaN
4,2007,08069888XG0003,AP069888XG,Approved,11/24/2006,11/24/2006,01/01/2007,No,No,United States,...,Yes,233307.9,233307.9,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [3]:
# taking out extra spaces
us_finance.columns = us_finance.columns.str.strip()

In [4]:
#dropping this columns
us_finance.drop(columns=['Multiyear Working Capital Extension', 'Loan Interest Rate','Working Capital Delegated Authority'], inplace=True)

In [5]:
#set to time
date_cols=['Decision Date','Effective Date','Expiration Date']
for cols in date_cols:
    us_finance[cols]=pd.to_datetime(us_finance[cols],errors='coerce')

In [6]:
#move "the" to the front
def move_the_to_front(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    if x.upper().endswith(" THE"):
        x = "The " + x[:-4].strip()

    return x

us_finance['Primary Exporter'] = us_finance['Primary Exporter'].apply(move_the_to_front)

In [7]:
us_finance['Primary Exporter'] = (
    us_finance['Primary Exporter']
    .str.upper()
    .str.strip()
    .str.replace(".", "", regex=False)
)

In [8]:
us_finance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52122 entries, 0 to 52121
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   Fiscal Year                            52122 non-null  int64         
 1   Unique Identifier                      52122 non-null  object        
 2   Deal Number                            52115 non-null  object        
 3   Decision                               52122 non-null  object        
 4   Decision Date                          52122 non-null  datetime64[ns]
 5   Effective Date                         50871 non-null  datetime64[ns]
 6   Expiration Date                        51435 non-null  datetime64[ns]
 7   Brokered                               52122 non-null  object        
 8   Deal Cancelled                         52122 non-null  object        
 9   Country                                52122 non-null  object

In [48]:
us_finance.isna().sum()/len(us_finance) *100

Fiscal Year                               0.000000
Unique Identifier                         0.000000
Deal Number                               0.013430
Decision                                  0.000000
Decision Date                             0.000000
Effective Date                            2.400138
Expiration Date                           1.318061
Brokered                                  0.000000
Deal Cancelled                            0.000000
Country                                   0.000000
Program                                   0.000000
Policy Type                              17.576072
Decision Authority                        0.013430
Primary Export Product NAICS/SIC code     2.889375
Product Description                       2.889375
Term                                      0.000000
Primary Applicant                         0.028779
Primary Lender                           58.742949
Primary Exporter                          0.372204
Primary Exporter City          

In [9]:
# I created a new column called country_only that contains just the country, because in the country column, it is mixed with
#Multiple - Countries", "Private Export Funding Corp, which are not actually countries, but we can't drop them cause of the size ans i seperated cause 
#it might affect our output for just country
us_finance["Country_Only"] = np.where(
    us_finance["Country"].isin(
        ["Multiple - Countries", "Private Export Funding Corp."]
    ),
    np.nan,
    us_finance["Country"]
)

In [10]:
us_finance.describe()

,Fiscal Year,Decision Date,Effective Date,Expiration Date,Approved/Declined Amount,Disbursed/Shipped Amount,Undisbursed Exposure Amount,Outstanding Exposure Amount,Small Business Authorized Amount,Woman Owned Authorized Amount,Minority Owned Authorized Amount
count,52122.000000,52122,50871,51435,5.212000e+04,5.212000e+04,5.212000e+04,5.212000e+04,5.212000e+04,5.212000e+04,5.212000e+04
mean,2014.653735,2014-12-06 05:44:30.887533056,2014-12-21 13:13:01.057970176,2015-12-14 20:56:28.976378112,5.062696e+06,4.278924e+06,2.971877e+05,3.381801e+05,1.177313e+06,8.322506e+04,1.354655e+05
min,2007.000000,2006-10-01 00:00:00,2005-02-15 00:00:00,2006-02-24 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2011.000000,2010-12-22 00:00:00,2010-12-28 00:00:00,2011-12-01 00:00:00,2.000000e+05,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+05,0.000000e+00,0.000000e+00
50%,2014.000000,2014-03-20 00:00:00,2014-04-01 00:00:00,2015-04-01 00:00:00,5.000000e+05,3.000000e+05,0.000000e+00,0.000000e+00,3.000000e+05,0.000000e+00,0.000000e+00
75%,2018.000000,2018-09-14 00:00:00,2018-10-29 12:00:00,2019-10-01 00:00:00,1.074673e+06,1.510491e+06,0.000000e+00,0.000000e+00,8.000000e+05,0.000000e+00,0.000000e+00
max,2025.000000,2025-09-30 00:00:00,2026-01-15 00:00:00,2030-05-21 00:00:00,3.595031e+09,2.942954e+09,2.046587e+09,2.344489e+09,3.332396e+08,4.000000e+07,3.600000e+07
std,4.963450,NaN,NaN,NaN,4.304295e+07,3.243132e+07,1.372011e+07,1.225901e+07,6.305852e+06,7.246259e+05,9.555308e+05


In [11]:
#spelling correction
us_finance['Product Description'] = us_finance['Product Description'].replace({
    'Radio and Television Broadcasting and Communicatio':
    'Radio and Television Broadcasting and Wireless Communications Equipment Manufacturing',

    'Radio and Television Broadcasting and Wireless Communications Equipment Man':
    'Radio and Television Broadcasting and Wireless Communications Equipment Manufacturing'
})

In [12]:
#trying to create a category
def classify_industry(product_description):
    if pd.isna(product_description):
        return 'Other'
    product = str(product_description).lower()

    # Aerospace & Defense
    if any(word in product for word in [
        'aircraft', 'helicopter', 'jet', 'air transportation',
        'airline', 'aircraft engine', 'aeronautical', 'navigation',
        'guidance', 'nautical system', 'guided missile', 'space vehicle',
        'aerospace', 'airport', 'flying field', 'flight training',
        'small arms', 'ordnance', 'ammunition', 'explosives'
    ]):
        return 'Aerospace & Defense'

    # Energy
    if any(word in product for word in [
        'oil', 'gas', 'petroleum', 'turbine', 'boiler', 'natural gas',
        'energy', 'electric power', 'coal', 'lubricating',
        'motor and generator', 'motors and generator', 'engine equipment',
        'power-driven', 'electric lamp', 'fluid power', 'water supply',
        'irrigation', 'air purification', 'mineral wool', 'refractory',
        'elevator', 'moving stairway', 'sewage', 'solid waste combustor',
        'incinerator', 'lime manufacturing'
    ]):
        return 'Energy'

    # Healthcare
    if any(word in product for word in [
        'medical', 'hospital', 'surgical', 'drug', 'pharmaceutical',
        'electromedical', 'medicinal', 'dental', 'irradiation', 'biologic',
        'diagnostic', 'ophthalmic', 'laboratory', 'cosmetic', 'beauty',
        'perfume', 'fire protection', 'offices of physicians',
        'health and personal care', 'optical goods', 'home health',
        'apiculture', 'veterinar'
    ]):
        return 'Healthcare'

    # Agriculture
    if any(word in product for word in [
        'farm', 'crop', 'grain', 'cotton', 'fruit', 'vegetable',
        'fertilizer', 'agricultural', 'flour', 'milling', 'dairy',
        'livestock', 'poultry', 'nursery', 'greenhouse', 'forestry',
        'lawn', 'garden', 'timber', 'logging', 'floriculture', 'soybean',
        'chicken egg', 'turkey production', 'animal production',
        'support activities for animal', 'forest nurseries', 'creamery',
        'butter', 'cheese', 'breakfast cereal', 'carpet and rug',
        'horse', 'equine', 'cattle', 'feedlot', 'apple orchard',
        'citrus', 'grape vineyard', 'mushroom', 'aquaculture',
        'prepared feeds', 'fluid milk', 'malt manufacturing'
    ]):
        return 'Agriculture'

    # Technology
    if any(word in product for word in [
        'semiconductor', 'semicinductor', 'computer', 'electronic',
        'electrical', 'communications', 'wireless', 'software', 'telephone',
        'audio', 'video', 'instrument', 'control device', 'relay', 'wiring',
        'transformer', 'switchgear', 'capacitor', 'resistor', 'printed circuit',
        'fiber optic', 'measuring', 'controlling device',
        'automatic environmental control', 'lighting', 'photographic',
        'photocopying', 'ball and roller bearing', 'copper wire',
        'totalizing fluid meter', 'counting device', 'watch', 'clock',
        'scale and balance', 'blank magnetic', 'optical recording',
        'computing infrastructure', 'data processing', 'web hosting',
        'cable and other', 'automatic controls for regulating'
    ]):
        return 'Technology'

    # Automotive
    if any(word in product for word in [
        'motor vehicle', 'automotive', 'passenger car', 'truck',
        'vehicle parts', 'transmission', 'automobile', 'motor home',
        'trailer', 'motorcycle', 'bicycle', 'tire', 'new car dealer',
        'used car dealer', 'recreational vehicle dealer'
    ]):
        return 'Automotive'

    # Transportation
    if any(word in product for word in [
        'railroad', 'rail', 'ship', 'transportation', 'freight',
        'boat', 'vessel', 'barge', 'ferry', 'tug', 'marine cargo',
        'packing and crating', 'warehousing', 'storage'
    ]):
        return 'Transportation'

    # Construction
    if any(word in product for word in [
        'construction', 'mining', 'drilling', 'excavat', 'cement',
        'concrete', 'asphalt', 'paving', 'specialty trade', 'plate work',
        'precision turned', 'building exterior contractor',
        'foundation, structure', 'site preparation', 'masonry',
        'building equipment contractor', 'building finishing contractor',
        'framing contractor', 'finish carpentry', 'residential remodel'
    ]):
        return 'Construction'

    # Building Materials
    if any(word in product for word in [
        'lumber', 'plywood', 'wood', 'millwork', 'sawmill', 'hardware',
        'plumbing', 'door', 'window', 'flooring', 'roofing', 'brick',
        'tile', 'glass', 'insulation', 'paint', 'wallboard',
        'building material', 'showcase', 'partition', 'shelving', 'locker',
        'mineral and earth', 'nonclay', 'primary aluminum',
        'aluminum rolling', 'aluminum extrud', 'cut stone', 'stone product',
        'home center', 'blind and shade', 'resilient floor',
        'curtain', 'drapery', 'iron foundries', 'custom roll forming',
        'copper rolling', 'drawing, extruding', 'nonferrous wire',
        'fastener', 'button', 'needle', 'saw blade', 'handsaw'
    ]):
        return 'Building Materials'

    # Chemicals
    if any(word in product for word in [
        'chemical', 'plastic', 'resin', 'coating', 'adhesive', 'rubber',
        'soap', 'detergent', 'polish', 'sanitation', 'cleaning', 'ink',
        'dye', 'pigment', 'synthetic', 'spice', 'extract', 'flavoring',
        'syrup', 'carbon and graphite', 'abrasive', 'surface active',
        'polystyrene', 'foam product', 'mayonnaise', 'dressing',
        'prepared sauce', 'ice manufacturing', 'tortilla', 'dry pasta',
        'specialty canning', 'broom', 'brush', 'mop', 'vacuum cleaner'
    ]):
        return 'Chemicals'

    # Food & Consumer Goods
    if any(word in product for word in [
        'food', 'meat', 'grocery', 'confectionery', 'wine', 'wineries',
        'tobacco', 'animal food', 'distiller', 'beverage', 'coffee', 'tea',
        'sugar', 'candy', 'bakery', 'snack', 'seafood', 'fish', 'frozen',
        'canned', 'bottled', 'breweries', 'pulp mill', 'caterer',
        'nondurable goods', 'kitchen utensil', 'cooking appliance',
        'household appliance', 'amusement', 'cookie', 'cracker',
        'commercial bakeries', 'retail bakeries', 'convenience store',
        'department store', 'warehouse club', 'supercenter',
        'other direct selling', 'pet and pet supplies', 'groceries',
    'groceries and related products',' Groceries - General Line','Groceries and Related Products - NEC'
    ]):
        return 'Food & Consumer Goods'

    # Textiles
    if any(word in product for word in [
        'yarn', 'textile', 'fiber', 'thread', 'apparel', 'clothing',
        'fabric', 'hosiery', 'knitting', 'weaving', 'leather', 'footwear',
        'shoe', 'outerwear', 'cut and sew', 'undergarment', 'luggage',
        'handbag', 'purse', 'canvas'
    ]):
        return 'Textiles'

    # Paper & Packaging
    if any(word in product for word in [
        'paper', 'paperboard', 'printing', 'publishing', 'cardboard',
        'envelope', 'stationery', 'book publish', 'bookbinding',
        'periodical', 'newsprint', 'greeting card', 'blankbook',
        'looseleaf', 'all other publishers', 'media representative'
    ]):
        return 'Paper & Packaging'

    # Recycling
    if any(word in product for word in [
        'recyclable', 'scrap', 'waste materials', 'waste management',
        'remediation', 'refuse', 'materials recovery', 'smelting',
        'secondary smelting', 'rolling - drawing', 'hazardous waste',
        'nonhazardous waste'
    ]):
        return 'Recycling'

    # Distribution & Trade
    if any(word in product for word in [
        'merchant wholesaler', 'merchant wholesalers', 'wholesale', 'broker',
        'durable goods', 'commercial equipment', 'international trade',
        'nondepository credit', 'holding compan', 'national commercial bank',
        'bank', 'professional equipment and supplies', 'commodity contracts',
        'trust', 'estate', 'agency account', 'pension fund',
        'consumer lending', 'sales financing', 'credit card',
        'federal and federally-sponsored credit', 'miscellaneous financial',
        'used merchandise', 'home furnishing store', 'all other consumer goods'
    ]):
        return 'Distribution & Trade'

    # Industrial Manufacturing
    if any(word in product for word in [
        'machinery', 'machine', 'metal', 'compressor', 'furnace', 'steel',
        'conveyor', 'pump', 'valve', 'crane', 'hoist', 'industrial',
        'welding', 'soldering', 'fabricat', 'forging', 'casting', 'stamping',
        'spring', 'bolt', 'nut', 'screw', 'wire product', 'heating',
        'air-conditioning', 'refrigerat', 'hvac', 'material handling',
        'power tool', 'hand tool', 'miscellaneous manufacturing',
        'manufacturing industries', 'die and tool', 'jig', 'fixture',
        'sign manufacturing', 'storage batter', 'actuator',
        'hand and edge tool', 'battery manufacturing', 'primary battery',
        'cutlery', 'flatware', 'manufactured home', 'mobile home'
    ]):
        return 'Industrial Manufacturing'

    # Consumer Goods
    if any(word in product for word in [
        'furniture', 'mattress', 'upholstered', 'cabinet', 'office furniture',
        'sport', 'athletic', 'toy', 'game', 'musical instrument', 'jewelry',
        'gift', 'jewelers', 'lapidary', 'musical groups', 'museums',
        'gambling', 'convention', 'trade show', 'language school',
        'real estate', 'commercial photography', 'advertising',
        'public relations', 'regulation, licensing'
    ]):
        return 'Consumer Goods'

    # Professional Services
    if any(word in product for word in [
        'service', 'services', 'management', 'surveying', 'mapping',
        'engineering', 'architect', 'consulting', 'research', 'testing',
        'repair', 'maintenance', 'offices of lawyers'
    ]):
        return 'Professional Services'

    # Administrative
    if any(word in product for word in [
        'nonclassifiable', 'placeholder'
    ]):
        return 'Administrative'

    return 'Other'

In [13]:
#mapping the new category
us_finance['Industry'] = us_finance['Product Description'].apply(classify_industry)

EXPLORATORY ANALYSIS

In [54]:
# Who receives U.S. export credit support? 
# Who receives the highest U.S. export financing?

exporter_support = (
    us_finance.groupby("Primary Exporter")["Approved/Declined Amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={
        "Primary Exporter": "Exporter",
        "Approved/Declined Amount": "Approved Amount"
    })
)

exporter_support["Approved Amount"] = exporter_support["Approved Amount"].apply(
    lambda x: f"${x:,.0f}"
)

print(exporter_support.head(10))

                                            Exporter  Approved Amount
0                                THE BOEING COMPANY,  $77,748,927,664
1                               MULTIPLE - EXPORTERS  $47,690,602,967
2                                BECHTEL CORPORATION   $5,193,143,361
3                     ANADARKO PETROLEUM CORPORATION   $5,000,000,000
4                               CBI AMERICAS LIMITED   $3,215,335,836
5                 GENERAL ELECTRIC ENERGY PARTS, INC   $3,097,175,496
6  GENERAL ELECTRIC INTERNATIONAL OPERATIONS COMP...   $3,060,329,341
7                                   EXXON MOBIL CORP   $3,005,400,000
8                        SOLAR TURBINES INCORPORATED   $2,720,899,619
9                                      OMATAPALO INC   $2,486,698,666


In [15]:
#Which industries benefit the most?
industry_support = (
    us_finance.groupby('Industry')['Approved/Declined Amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

industry_support.columns = ['Industry', 'Approved Amount']

industry_support['Approved Amount'] = industry_support['Approved Amount'].apply(
    lambda x: f'${x:,.0f}'
)

print(industry_support)

                    Industry  Approved Amount
0        Aerospace & Defense  $87,680,661,478
1                      Other  $46,557,631,469
2                     Energy  $35,207,433,688
3      Professional Services  $16,692,108,329
4                 Technology  $13,749,669,469
5   Industrial Manufacturing  $12,277,737,685
6                Agriculture   $8,719,804,315
7                 Automotive   $6,290,453,518
8      Food & Consumer Goods   $5,302,602,374
9               Construction   $5,175,147,037
10                 Chemicals   $4,722,415,316
11            Transportation   $4,388,426,081
12        Building Materials   $3,822,445,564
13            Administrative   $3,445,480,559
14                Healthcare   $2,799,222,495
15      Distribution & Trade   $2,585,999,079
16                  Textiles   $2,407,885,639
17         Paper & Packaging   $1,148,224,446
18                 Recycling     $506,600,021
19            Consumer Goods     $387,747,109


In [66]:
# Are approvals concentrated among a small number of companies?
exporter_support = (
    us_finance[
        (us_finance["Primary Exporter"] != "MULTIPLE - EXPORTERS") &
        (us_finance["Primary Exporter"].notna())
    ]
    .groupby("Primary Exporter")["Approved/Declined Amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

total = exporter_support["Approved/Declined Amount"].sum()

exporter_support["Share (%)"] = (
    exporter_support["Approved/Declined Amount"] / total * 100
)

exporter_support["Cumulative Share (%)"] = (
    exporter_support["Share (%)"].cumsum()
)

# Format the Approved Amount column as currency
exporter_support["Approved/Declined Amount"] = exporter_support["Approved/Declined Amount"].apply(
    lambda x: f"${x:,.0f}"
)

exporter_support.head(10)

,Primary Exporter,Approved/Declined Amount,Share (%),Cumulative Share (%)
0,"THE BOEING COMPANY,","$77,748,927,664",36.415566,36.415566
1,BECHTEL CORPORATION,"$5,193,143,361",2.432333,38.847898
2,ANADARKO PETROLEUM CORPORATION,"$5,000,000,000",2.341869,41.189768
3,CBI AMERICAS LIMITED,"$3,215,335,836",1.505979,42.695747
4,"GENERAL ELECTRIC ENERGY PARTS, INC","$3,097,175,496",1.450636,44.146383
5,GENERAL ELECTRIC INTERNATIONAL OPERATIONS COMP...,"$3,060,329,341",1.433378,45.579761
6,EXXON MOBIL CORP,"$3,005,400,000",1.407651,46.987412
7,SOLAR TURBINES INCORPORATED,"$2,720,899,619",1.274398,48.261811
8,OMATAPALO INC,"$2,486,698,666",1.164705,49.426515
9,APPLIED MATERIALS INC,"$2,126,638,181",0.996062,50.422577


In [16]:
#How has the total value of U.S. export credit approvals changed over time?
approval_trend = (
    us_finance.groupby('Fiscal Year')['Approved/Declined Amount']
    .sum()
    .reset_index()
)

approval_trend.columns = ['Fiscal Year', 'Approved Amount']

approval_trend['Approved Amount'] = approval_trend['Approved Amount'].apply(
    lambda x: f'${x:,.0f}'
)
approval_trend

,Fiscal Year,Approved Amount
0,2007,"$12,567,363,445"
1,2008,"$14,395,443,704"
2,2009,"$21,021,115,995"
3,2010,"$24,467,802,703"
4,2011,"$32,727,089,509"
5,2012,"$35,797,343,173"
6,2013,"$27,396,060,658"
7,2014,"$20,482,897,764"
8,2015,"$12,699,952,541"
9,2016,"$5,037,093,109"


In [18]:
#What percentage of approved financing is actually disbursed?
overall_disbursement_rate = (
    us_finance['Disbursed/Shipped Amount'].sum()
    / us_finance['Approved/Declined Amount'].sum()
    * 100
)

print(f"{overall_disbursement_rate:.2f}%")

84.52%


In [19]:
#Do deals involving brokers have higher approval rates compared to direct applications?
broker_analysis = (
    us_finance.groupby('Brokered')
    .agg(
        Total_Deals=('Decision', 'count'),
        Approved_Deals=('Decision',
                        lambda x: (x.str.lower() == 'approved').sum())
    )
)

broker_analysis['Approval Rate (%)'] = (
    broker_analysis['Approved_Deals']
    / broker_analysis['Total_Deals']
    * 100
).round(2)

print(broker_analysis)

          Total_Deals  Approved_Deals  Approval Rate (%)
Brokered                                                
No              12129           12126              99.98
Yes             39993           39990              99.99


In [68]:
# Which export credit programs convert approved financing into actual exports most successfully?

program_performance = (
    us_finance.groupby("Program")[
        ["Approved/Declined Amount", "Disbursed/Shipped Amount"]
    ]
    .sum()
    .reset_index()
)

# Calculate Disbursement Rate
program_performance["Disbursement Rate (%)"] = (
    program_performance["Disbursed/Shipped Amount"]
    / program_performance["Approved/Declined Amount"]
) * 100

# Round percentage
program_performance["Disbursement Rate (%)"] = (
    program_performance["Disbursement Rate (%)"].round(2)
)

# Format currency columns
program_performance["Approved/Declined Amount"] = (
    program_performance["Approved/Declined Amount"]
    .apply(lambda x: f"${x:,.0f}")
)

program_performance["Disbursed/Shipped Amount"] = (
    program_performance["Disbursed/Shipped Amount"]
    .apply(lambda x: f"${x:,.0f}")
)

# Sort by highest disbursement rate
program_performance = program_performance.sort_values(
    by="Disbursement Rate (%)",
    ascending=False
)

program_performance

,Program,Approved/Declined Amount,Disbursed/Shipped Amount,Disbursement Rate (%)
3,Working Capital,"$29,661,709,855","$27,476,874,201",92.63
1,Insurance,"$73,049,980,045","$64,073,119,167",87.71
0,Guarantee,"$118,135,883,569","$100,557,858,918",85.12
2,Loan,"$43,020,122,203","$30,909,643,882",71.85


In [21]:
us_finance

,Fiscal Year,Unique Identifier,Deal Number,Decision,Decision Date,Effective Date,Expiration Date,Brokered,Deal Cancelled,Country,...,Primary Source of Repayment (PSOR),Approved/Declined Amount,Disbursed/Shipped Amount,Undisbursed Exposure Amount,Outstanding Exposure Amount,Small Business Authorized Amount,Woman Owned Authorized Amount,Minority Owned Authorized Amount,Country_Only,Industry
0,2007,08003048XA0002,AP003048AA,Approved,2006-11-29,NaT,2011-12-15,No,No,Private Export Funding Corp.,...,Various Us Companies,24500000.0,24500000.0,0.0,0.0,0.0,0.0,0.0,NaN,Other
1,2007,08003048XA0003,AP003048AA,Approved,2006-11-29,2006-11-29,2016-12-15,No,No,Private Export Funding Corp.,...,Various Us Companies,50000000.0,50000000.0,0.0,0.0,0.0,0.0,0.0,NaN,Other
2,2007,08003048XA0004,AP003048AA,Approved,2007-08-16,2007-08-16,2017-09-15,No,No,Private Export Funding Corp.,...,Various Us Companies,136250000.0,136250000.0,0.0,0.0,0.0,0.0,0.0,NaN,Other
3,2007,08064737XM0001,AP064737XM,Approved,2007-09-21,2007-09-21,2008-07-31,No,No,United States,...,American General Supplies,2700000.0,2700000.0,0.0,0.0,2700000.0,0.0,2700000.0,United States,Transportation
4,2007,08069888XG0003,AP069888XG,Approved,2006-11-24,2006-11-24,2007-01-01,No,No,United States,...,Kell-Strom Tool Co Inc,233307.9,233307.9,0.0,0.0,0.0,0.0,0.0,United States,Energy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52117,2025,09787003ST1001,787003,Approved,2025-09-19,2025-11-15,2026-12-01,Yes,No,Mexico,...,Imporagri SA De CV,98000.0,0.0,98000.0,0.0,98000.0,0.0,0.0,Mexico,Agriculture
52118,2025,09787330ST1001,787330,Approved,2025-09-17,2025-11-01,2026-11-01,Yes,No,Costa Rica,...,Polyflex SA,45000.0,0.0,45000.0,0.0,45000.0,0.0,45000.0,Costa Rica,Chemicals
52119,2025,09787377ST1001,787377,Approved,2025-09-24,2025-09-01,2026-09-01,Yes,No,Multiple - Countries,...,Multiple - PSORs,700000.0,0.0,700000.0,0.0,0.0,0.0,0.0,NaN,Food & Consumer Goods
52120,2025,09787399ST1001,787399,Approved,2025-09-22,2025-10-01,2026-10-01,Yes,No,Canada,...,Atlantic Air Cleaning Specialists LTD,36000.0,0.0,36000.0,0.0,36000.0,36000.0,0.0,Canada,Industrial Manufacturing


In [22]:
us_finance.to_csv("us_finance.csv", index=False)